In [4]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import cv2
import os
import sys
import pandas as pd

sys.path.append('/media/edward/HDD/Workspace/lang-segment-anything/')

import random
from PIL import Image
from lang_sam import LangSAM
from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator
from ultralytics import YOLO

# Check device availability
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

# Clear the CUDA memory cache
torch.cuda.empty_cache()

# Load SAM2 model checkpoint and config
sam2_checkpoint = "../checkpoints/sam2.1_hiera_small.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_s.yaml"
model = YOLO('/media/edward/HDD/Docker-Workspace/YOLOv8/runs/classify/train32/weights/best.pt')

LAM_model = LangSAM()

colors = {
    'Leaf': (255, 0, 0),    # Blue
    'Panicle': (0, 0, 255), # Red
    'Noise': (0, 255, 255),  # Yellow
    'PanicleClump': (128, 0, 128),    # Purple
}

# Build the SAM2 model
sam2_model = build_sam2(model_cfg, sam2_checkpoint, device=device)

# Create an automatic mask generator with custom parameters
mask_generator = SAM2AutomaticMaskGenerator(
    model=sam2_model,
    points_per_side=64, 
    pred_iou_thresh=0.80,
    stability_score_thresh=0.90,
    crop_n_layers=2,
    crop_n_points_downscale_factor=1,
    min_mask_region_area=100,
)

def read_png_with_alpha(image_path):
    image = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)
    if image is None:
        raise ValueError(f"Error loading image at {image_path}")
    
    image = np.array(image, copy=True)  # Ensure it is writable
    if image.shape[2] == 4:
        bgr_channels = image[:, :, :3]
        alpha_channel = image[:, :, 3]
        bgr_channels[alpha_channel == 0] = (0, 0, 0)
        image = cv2.merge([bgr_channels, alpha_channel])
    return image


def checkcolour(masks, hsv):
    colours = np.zeros((0,3))
    for i in range(len(masks)):
        color = hsv[masks[i]['segmentation']].mean(axis=(0))
        colours = np.append(colours, color[None, :], axis=0)
    idx_green = (colours[:,0]<85) & (colours[:,0]>5) & ((colours[:,2]>90) | ((colours[:,1]>15) & (colours[:,2]>45)))
    return idx_green

def add_labels_to_masks(npz_data, original_image):
    original_h, original_w = original_image.shape[:2]
    masks = npz_data['arr_0']
    panicle_count = 0
    for i, mask in enumerate(masks):
        segmentation_mask = mask['segmentation'].astype(np.uint8)

        # Resize the segmentation mask to match the original image dimensions
        resized_segmentation_mask = cv2.resize(segmentation_mask, (original_w, original_h), interpolation=cv2.INTER_NEAREST)
        
        # Find contours in the resized segmentation mask
        contours, _ = cv2.findContours(resized_segmentation_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if contours:
            # Select the largest contour by area
            largest_contour = max(contours, key=cv2.contourArea)

            # Create a new binary mask with only the largest contour
            largest_contour_mask = np.zeros_like(resized_segmentation_mask, dtype=np.uint8)
            cv2.drawContours(largest_contour_mask, [largest_contour], -1, color=255, thickness=cv2.FILLED)

            # Update the mask with only the largest contour
            mask['segmentation'] = largest_contour_mask

            # Visualize the updated mask (optional)
            # plt.figure(figsize=(6, 6))
            # plt.imshow(largest_contour_mask, cmap='gray')
            # plt.title(f"Mask {i+1} with Only Largest Contour")
            # plt.axis('off')
            # plt.show()

        # Create a masked image using the updated segmentation mask
        masked_image = cv2.bitwise_and(original_image, original_image, mask=mask['segmentation'])

        # Prepare the image for YOLO by cropping and resizing
        x, y, w, h = cv2.boundingRect(largest_contour)
        if w * h < 500:  # Skip very small regions
            continue

        cropped_image = masked_image[y:y+h, x:x+w]
        resized_padded_image = resize_and_pad(cropped_image)
        # Make predictions with YOLO
        resized_padded_image_rgb = cv2.cvtColor(resized_padded_image, cv2.COLOR_BGR2RGB)
        prediction = model.predict(resized_padded_image_rgb, verbose=False)


        if prediction and hasattr(prediction[0], 'probs') and hasattr(prediction[0].probs, 'data'):
            try:
                # Access the data tensor and get the index of the highest probability class
                max_index = prediction[0].probs.data.argmax().item()
                label = prediction[0].names[max_index]  # Extract the label
                confidence = prediction[0].probs.data[max_index].item()  # Extract the confidence
                # prediction[0].show()
                # Handle low-confidence cases
                if label == "Leaf" and confidence < 0.99:
                    label = "Panicle"
                    #print(f"Low confidence for 'Leaf', changed to 'Panicle' (confidence: {confidence*100:.2f}%)")
                #else:
                    #print(f"Detected: {label} with {confidence*100:.2f}% confidence")
                    
                if label == "Panicle":
                    panicle_count = panicle_count+1
                #else:
                    #print(f"Detected: {label} with {confidence*100:.2f}% confidence")
                
            except Exception as e:
                print(f"Error while processing prediction: {e}")
                label = 'No Detection'
                
            # print(f"Prediction object: {prediction[0]}")
            # print(f"Predicted probabilities: {prediction[0].probs}")
            # plt.figure(figsize=(6, 6))
            # plt.imshow(resized_padded_image)
            # plt.title(f"Predicted: {label} ({confidence*100:.2f}%)")
            # plt.axis('off')
            # plt.show()
        else:
            label = 'No Detection'

        # Assign the label to the mask
        mask['label'] = label
    # print(f"Number of Panicles counted: {panicle_count}")
    # plt.figure(figsize=(6, 6))
    # plt.imshow(cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB))
    # plt.axis('off')
    # plt.show()
    return masks


def visualize_masks_with_classes(original_image, masks, depth_raw_path):
    overlay = np.zeros_like(original_image, dtype=np.uint8)
    # overlay the masks on the canvas
    for mask in masks:
        segmentation_mask = mask['segmentation'].astype(np.uint8)
        #resized_segmentation_mask = cv2.resize(segmentation_mask, (original_w, original_h), interpolation=cv2.INTER_NEAREST)
        label = mask.get('label', 'No Label')
        
        # Get color corresponding to the label
        color = colors.get(label, (255, 255, 255))  # default to white if unknown
        #print(f"Label: {label}, color: {color}")
        
        contours, _ = cv2.findContours(segmentation_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(overlay, contours, -1, color, thickness=2)
        
        overlay[segmentation_mask> 0] = (overlay[segmentation_mask > 0] * 0.5 + np.array(color) * 0.5).astype(np.uint8)
    
    #overlay
    output_image = cv2.addWeighted(original_image, 0.5, overlay, 0.5, 0)
    write_folder = depth_raw_path.replace('_depth_raw', '_yolo_overlay')
    
    cv2.imwrite(write_folder, output_image)
    # Display the result
    # plt.figure(figsize=(10, 5))
    # plt.imshow(cv2.cvtColor(output_image, cv2.COLOR_BGR2RGB))

    # plt.title('Overlayed Masks with YOLOv8 Labels')
    # plt.axis('off')
    # plt.show()
    
def process_leaf_masks_on_depth(original_bgr_image, depth_raw_path, results_folder_name, masks):
    depth_image = cv2.imread(depth_raw_path, cv2.IMREAD_GRAYSCALE)
    if depth_image is None:
        raise ValueError(f"Error: Image at '{depth_raw_path}' could not be loaded. Check the file path.")
    
    depth_image_resized = cv2.resize(depth_image, (original_bgr_image.shape[1], original_bgr_image.shape[0]))
    
    # Step 1: Identify masks labeled as 'Leaf' and calculate maximum intensity for each mask
    leaf_masks = []
    for i, mask in enumerate(masks):
        if mask.get('label') == 'Leaf':
            segmentation_mask = mask['segmentation'].astype(np.uint8)
            masked_depth_image = cv2.bitwise_and(depth_image_resized, depth_image_resized, mask=segmentation_mask)
            max_intensity = masked_depth_image.max()
            leaf_masks.append((i, max_intensity))
    
    # Step 2: Filter masks to keep only those in the top 10% of max intensities
    if leaf_masks:
        intensities = [data[1] for data in leaf_masks]
        intensity_threshold = np.percentile(intensities, 90)  # Get the 90th percentile
        filtered_leaf_masks = [(i, intensity) for i, intensity in leaf_masks if intensity >= intensity_threshold]
    
        # Step 3: Calculate the area for each of the filtered masks and select the top 3 largest masks
        area_data = []
        for i, intensity in filtered_leaf_masks:
            mask = masks[i]
            segmentation_mask = mask['segmentation'].astype(np.uint8)
            area = cv2.countNonZero(segmentation_mask)
            area_data.append((i, area))
        
        top_leaf_masks = sorted(area_data, key=lambda x: x[1], reverse=True)[:3]
    
        # Step 4: Select the mask with the highest maximum intensity from the top 3 largest masks
        max_intensity_data = []
        for i, area in top_leaf_masks:
            mask = masks[i]
            segmentation_mask = mask['segmentation'].astype(np.uint8)
            #resized_segmentation_mask = cv2.resize(segmentation_mask, (depth_image_resized.shape[1], depth_image_resized.shape[0]), interpolation=cv2.INTER_NEAREST)
            masked_depth_image = cv2.bitwise_and(depth_image_resized, depth_image_resized, mask=segmentation_mask)
            max_intensity = masked_depth_image.max()
            max_intensity_data.append((i, max_intensity))
    
        # Step 5: Select the mask with the highest maximum intensity from the top 3 largest masks
        if max_intensity_data:
            mask_index, max_intensity = max(max_intensity_data, key=lambda x: x[1])
            mask = masks[mask_index]
            segmentation_mask = mask['segmentation'].astype(np.uint8)
            #resized_segmentation_mask = cv2.resize(segmentation_mask, (original_rgb_image.shape[1], original_rgb_image.shape[0]), interpolation=cv2.INTER_NEAREST)
            
            # Mask the original RGB image
            masked_bgr_image = cv2.bitwise_and(original_bgr_image, original_bgr_image, mask=segmentation_mask)
            write_folder = depth_raw_path.replace('_depth_raw', results_folder_name)
            print(f"Saving isolated leaf image at {write_folder}")
            cv2.imwrite(write_folder, masked_bgr_image)
            return
        
    print("No leaf masks found in the image.")
    black_image = np.zeros_like(original_bgr_image)
    write_folder = depth_raw_path.replace('_depth_raw', results_folder_name)
    cv2.imwrite(write_folder, black_image)

def resize_and_pad(image):
    try:
        #print(f"Original image shape before resizing: {image.shape}")
        h, w = image.shape[:2]
        if h > w:
            new_h = 640
            new_w = int(w * (640 / h))
        else:
            new_w = 640
            new_h = int(h * (640 / w))
        #print(f"New dimensions: (new_h={new_h}, new_w={new_w})")
        
        resized_image = cv2.resize(image, (new_w, new_h))
        pad_vertical = (640 - new_h) // 2
        pad_horizontal = (640 - new_w) // 2
        #print(f"Padding: vertical={pad_vertical}, horizontal={pad_horizontal}")
        
        padded_image = cv2.copyMakeBorder(
            resized_image, pad_vertical, pad_vertical, pad_horizontal, pad_horizontal,
            cv2.BORDER_CONSTANT, value=[0, 0, 0]
        )
        #print(f"Padded image shape: {padded_image.shape}")
        return padded_image
    except Exception as e:
        print(f"Error in resize_and_pad: {e}")
        raise


def process_images(image_paths, results_folder_name):
    
    for image_path in image_paths:
        npz_path = os.path.splitext(image_path)[0] + '_SAM2.npz'
        npz_path = npz_path.replace('depth_seg', 'npz_v11')
        npz_data = np.load(npz_path, allow_pickle=True)
        depth_raw_path = image_path.replace('masked_', 'depth_').replace('depth_seg', 'depth_raw')
        check_img = depth_raw_path.replace('_depth_raw', results_folder_name)
        lam_masked_path = depth_raw_path.replace('_depth_raw', '_results_rgb_11/lam')
        if os.path.exists(check_img):
            print(f"Skipping {check_img}, already processed.")
            continue

        original_image = read_png_with_alpha(image_path)

        # Initialize the LangSAM model
        original_image = retrieve_masked_section(original_image, lam_masked_path)

        

        masks = add_labels_to_masks(npz_data, original_image)
        visualize_masks_with_classes(original_image, masks, depth_raw_path)
        process_leaf_masks_on_depth(original_image, depth_raw_path, results_folder_name, masks)


def process_segmentation_masks(depth_seg_folder, npz_folder, date):
    depth_seg_names = [x for x in os.listdir(depth_seg_folder) if '.png' in x or '.jpg' in x]
    depth_seg_paths = [os.path.join(depth_seg_folder, x) for x in depth_seg_names]
    #print(depth_seg_paths)

    for imname in depth_seg_names:
        #define the corresponding .npz filename
        if imname.endswith('.png'):
            npz_file_path = os.path.join(npz_folder, imname.replace('.png', '_SAM2.npz'))
        elif imname.endswith('.jpg'):
            npz_file_path = os.path.join(npz_folder, imname.replace('.jpg', '_SAM2.npz'))


        #check if the .npz file already exists, skip processing if it does - this takes hours to run so want to be able to handle interruptions
        if os.path.exists(npz_file_path):
            print(f"Skipping {imname}, already processed.")
            continue  # Skip to the next image

        #print(f"Processing {imname}")
        image = read_png_with_alpha(os.path.join(depth_seg_folder, imname)) # if depth segmented image is png, SAM ignores alpha channel and draws masks on background also
        if image is None:
            print(f"Error loading image {os.path.join(depth_seg_folder, imname)}")
            continue

        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

        #get masks, this took approx 100 second son a 4090 in a docker container
        masks = mask_generator.generate(image)

        # kept the checkColour from leaf only sam, edited it to suit oats and our specific better
        idx_green = checkcolour(masks, hsv)

        masks_g = []
        for idx, use in enumerate(idx_green):
            if use:
                masks_g.append(masks[idx])

        # Save results as npz file
        handle_overlapping_masks(masks_g)   #TODO
        np.savez(npz_file_path, masks_g)
        print(f"Done processing npz masks for {imname} in {date}")
            
def handle_overlapping_masks(masks):
    to_remove = []  # List to track indices of masks to remove

    for i in range(len(masks)):
        for j in range(i + 1, len(masks)):
            mask_i = masks[i]['segmentation'].astype(np.uint8)
            mask_j = masks[j]['segmentation'].astype(np.uint8)

            # Calculate overlap between masks
            overlap = cv2.bitwise_and(mask_i, mask_j)
            if np.any(overlap):
                # Calculate areas of masks
                area_i = np.sum(mask_i)
                area_j = np.sum(mask_j)

                # Subtract the smaller mask from the larger mask
                if area_i > area_j:
                    updated_mask_i = mask_i - overlap
                    updated_mask_j = mask_j
                else:
                    updated_mask_i = mask_i
                    updated_mask_j = mask_j - overlap
                

                # Update the masks
                masks[i]['segmentation'] = updated_mask_i
                masks[j]['segmentation'] = updated_mask_j

                # Check for significant area loss and mark for removal if needed
                if np.sum(updated_mask_i) < 0.1 * area_i or np.sum(updated_mask_i) < 500:
                    to_remove.append(i)
                if np.sum(updated_mask_j) < 0.1 * area_j or np.sum(updated_mask_j) < 500:
                    to_remove.append(j)

    # Ensure all masks remain binary and remove masks with significant area loss
    masks = [mask for idx, mask in enumerate(masks) if idx not in set(to_remove)]
    for mask in masks:
        mask['segmentation'] = (mask['segmentation'] > 0).astype(np.uint8)

    return masks


def visualize_npz_masks(npz_folder, output_folder):
    """
    Visualize masks from all NPZ files in the specified folder with random colors.

    Args:
        npz_folder (str): Path to the folder containing NPZ files.
        output_folder (str): Path to save the visualization outputs.
    """
    # Ensure the output folder exists
    os.makedirs(output_folder, exist_ok=True)

    # Get list of NPZ files
    npz_files = [f for f in os.listdir(npz_folder) if f.endswith('.npz')]

    for npz_file in npz_files:
        npz_path = os.path.join(npz_folder, npz_file)
        npz_data = np.load(npz_path, allow_pickle=True)
        masks = npz_data['arr_0']

        # Create a blank canvas
        first_mask = masks[0]['segmentation']
        canvas_h, canvas_w = first_mask.shape
        canvas = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)

        # Assign random colors to each mask
        for mask in masks:
            segmentation_mask = mask['segmentation'].astype(np.uint8)
            random_color = [random.randint(0, 255) for _ in range(3)]

            # Apply the color to the canvas
            canvas[segmentation_mask > 0] = canvas[segmentation_mask > 0] * 0.5 + np.array(random_color) * 0.5

        # Save and display the visualization
        output_image_path = os.path.join(output_folder, npz_file.replace('.npz', '_visualization.png'))
        # plt.figure(figsize=(10, 5))
        # plt.imshow(canvas)
        # plt.title(f'Mask Visualization: {npz_file}')
        # plt.axis('off')
        # plt.savefig(output_image_path)
        # plt.close()

        print(f"Saved visualization for {npz_file} at {output_image_path}")


def retrieve_masked_section(image, lam_path):
    box_threshold = 0.2
    text_threshold = 0
    pil_image = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    # Perform prediction
    results = LAM_model.predict(
        [pil_image],
        ["oat plant"],
        box_threshold=box_threshold,
        text_threshold=text_threshold,
    )
    masks = results[0]["masks"]
    image_width, image_height = pil_image.size
    image_center = np.array([image_width / 2, image_height / 2])

    # Calculate centroids of all masks
    centroids = []
    for mask in masks:
        # Find coordinates of all non-zero pixels
        y_coords, x_coords = np.where(mask > 0)
        centroid = np.array([np.mean(x_coords), np.mean(y_coords)])
        centroids.append(centroid)

    centroids = np.array(centroids)

    # Find the mask closest to the image center
    distances = np.linalg.norm(centroids - image_center, axis=1)
    closest_idx = np.argmin(distances)

    # Extract the corresponding mask
    selected_mask = masks[closest_idx].astype(np.uint8)

    # Calculate the mask area (number of pixels > 0 in the mask)
    mask_area = np.sum(selected_mask > 0)  # Count all non-zero pixels in the mask

    # Perform bitwise AND to check overlap
    binary_and_result = cv2.bitwise_and(image, image, mask=selected_mask)
    non_zero_pixels = cv2.countNonZero(cv2.cvtColor(binary_and_result, cv2.COLOR_BGR2GRAY))

    # Compare overlap ratio to mask area
    ratio = non_zero_pixels / mask_area if mask_area > 0 else 0

    if ratio < 0.1:  # If less than 10% of the mask overlaps with the object
        print(f"Detected an inverted mask, saving the original image at {lam_path}")
        cv2.imwrite(lam_path, image)
        return image

    # Apply the mask to the image
    image_array = np.asarray(image)  # Convert PIL image to NumPy array
    masked_image_array = np.zeros_like(image_array)
    for channel in range(3):  # Assuming RGB
        masked_image_array[:, :, channel] = image_array[:, :, channel] * selected_mask


    # Save the result
    print(f"Saving lam image at {lam_path}")
    cv2.imwrite(lam_path, masked_image_array)

    return masked_image_array






Using device: cuda


In [5]:
base_dir = "/media/edward/HDD/Docker-Workspace/Data/GH4-Completed"
dates = ['2024-08-29-Evening',
         '2024-08-29-Morning',
         '2024-09-02-Evening',
         '2024-09-03-Evening',
         '2024-09-03-Morning',
         '2024-09-04-Morning'
        ]
for date in dates:
    depth_seg_folder = os.path.join(base_dir, date, date + '_depth_seg/')
    print(depth_seg_folder)
    results_folder_name = '_results_rgb_11/images'
    results_folder = os.path.join(base_dir, date, date + results_folder_name)
    lam_folder = os.path.join(base_dir, date, date + '_results_rgb_11/lam')
    npz_folder = os.path.join(base_dir, date, date + '_npz_v11/')
    os.makedirs(results_folder, exist_ok=True)
    os.makedirs(npz_folder, exist_ok=True)
    os.makedirs(lam_folder, exist_ok=True)
    depth_seg_names = [x for x in os.listdir(depth_seg_folder) if x.endswith(('.png', '.jpg'))]
    depth_seg_paths = [os.path.join(depth_seg_folder, x) for x in depth_seg_names]
    print(depth_seg_paths)
    process_segmentation_masks(depth_seg_folder, npz_folder, date)
    process_images(depth_seg_paths, results_folder_name)


/media/edward/HDD/Docker-Workspace/Data/GH4-Completed/2024-08-29-Evening/2024-08-29-Evening_depth_seg/
['/media/edward/HDD/Docker-Workspace/Data/GH4-Completed/2024-08-29-Evening/2024-08-29-Evening_depth_seg/masked_N-2-3_rgb_cam18443010715BEE0F00_29_08_2024_16_45_30_X3678.7_Y1033.0_Z690.1_RX180.0_RY0.0_RZ0.0_49.jpg', '/media/edward/HDD/Docker-Workspace/Data/GH4-Completed/2024-08-29-Evening/2024-08-29-Evening_depth_seg/masked_D-12-5_rgb_cam19443010D1BE671300_29_08_2024_15_41_54_X605.0_Y721.0_Z635.0_RX-120.0_RY90.0_RZ180.0_56.jpg', '/media/edward/HDD/Docker-Workspace/Data/GH4-Completed/2024-08-29-Evening/2024-08-29-Evening_depth_seg/masked_D-11-1_rgb_cam19443010D1BE671300_29_08_2024_16_36_33_X3497.2_Y1853.2_Z635.0_RX-120.0_RY90.0_RZ0.0_56.jpg', '/media/edward/HDD/Docker-Workspace/Data/GH4-Completed/2024-08-29-Evening/2024-08-29-Evening_depth_seg/masked_D-5-4_rgb_cam19443010D1BE671300_29_08_2024_16_23_06_X2801.1_Y1027.0_Z635.0_RX-120.0_RY90.0_RZ0.0_56.jpg', '/media/edward/HDD/Docker-Worksp

/media/edward/HDD/anaconda3/envs/full_oat_pipeline/lib/python3.11/site-packages/torchvision/transforms/functional.py:154: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at ../torch/csrc/utils/tensor_numpy.cpp:206.)
  img = torch.from_numpy(pic.transpose((2, 0, 1))).contiguous()


Saving isolated leaf image at /media/edward/HDD/Docker-Workspace/Data/GH4-Completed/2024-09-02-Evening/2024-09-02-Evening_results_rgb_11/images/depth_D-12-3_rgb_cam19443010D1BE671300_02_09_2024_15_32_58_X4485.2_Y705.0_Z635.0_RX-120.0_RY90.0_RZ0.0_56.jpg
Predicting 1 masks
Predicted 1 masks
Saving lam image at /media/edward/HDD/Docker-Workspace/Data/GH4-Completed/2024-09-02-Evening/2024-09-02-Evening_results_rgb_11/lam/depth_D-6-1_rgb_cam19443010D1BE671300_02_09_2024_14_20_15_X784.8_Y1836.0_Z635.0_RX-120.0_RY90.0_RZ0.0_56.jpg
Saving isolated leaf image at /media/edward/HDD/Docker-Workspace/Data/GH4-Completed/2024-09-02-Evening/2024-09-02-Evening_results_rgb_11/images/depth_D-6-1_rgb_cam19443010D1BE671300_02_09_2024_14_20_15_X784.8_Y1836.0_Z635.0_RX-120.0_RY90.0_RZ0.0_56.jpg
Predicting 1 masks
Predicted 1 masks
Detected an inverted mask, saving the original image at /media/edward/HDD/Docker-Workspace/Data/GH4-Completed/2024-09-02-Evening/2024-09-02-Evening_results_rgb_11/lam/depth_N-9-5_

In [6]:
visualize_npz_masks(npz_folder, results_folder)

Saved visualization for masked_N-1-3_rgb_cam19443010D1BE671300_04_09_2024_09_22_06_X2776.2_Y694.2_Z635.0_RX-120.0_RY90.0_RZ180.0_56_SAM2.npz at /media/edward/HDD/Docker-Workspace/Data/GH4-Completed/2024-09-04-Morning/2024-09-04-Morning_results_rgb_11/images/masked_N-1-3_rgb_cam19443010D1BE671300_04_09_2024_09_22_06_X2776.2_Y694.2_Z635.0_RX-120.0_RY90.0_RZ180.0_56_SAM2_visualization.png
Saved visualization for masked_D-10-3_rgb_cam18443010715BEE0F00_04_09_2024_09_23_28_X3061.3_Y1458.0_Z690.1_RX180.0_RY0.0_RZ180.0_49_SAM2.npz at /media/edward/HDD/Docker-Workspace/Data/GH4-Completed/2024-09-04-Morning/2024-09-04-Morning_results_rgb_11/images/masked_D-10-3_rgb_cam18443010715BEE0F00_04_09_2024_09_23_28_X3061.3_Y1458.0_Z690.1_RX180.0_RY0.0_RZ180.0_49_SAM2_visualization.png
Saved visualization for masked_N-1-4_rgb_cam18443010715BEE0F00_04_09_2024_08_36_16_X479.7_Y672.0_Z690.1_RX180.0_RY0.0_RZ0.0_49_SAM2.npz at /media/edward/HDD/Docker-Workspace/Data/GH4-Completed/2024-09-04-Morning/2024-09-04